# Strategy - Volume Flow

In [ ]:
# ============================================================
# S05 - Volume Flow
# ============================================================

import xarray as xr
import numpy as np

import qnt.data as qndata
import qnt.ta as qnta
import qnt.output as qnout

# ============================================================
# Load Data
# ============================================================

data = qndata.cryptodaily_load_data(
    min_date="2015-01-01"
)

# ============================================================
# Strategy
# ============================================================

def strategy(data):

    close = data.sel(field="close")
    vol = data.sel(field="vol")
    is_liquid = data.sel(field="is_liquid")

    # --------------------------------------------------------
    # Daily Returns
    # --------------------------------------------------------

    ret = close / close.shift(time=1) - 1

    vol20 = qnta.std(ret, 20)

    # --------------------------------------------------------
    # Volume Flow Signal
    # --------------------------------------------------------

    vol_ma20 = qnta.sma(vol, 20)

    volume_ratio = vol / (vol_ma20 + 1e-6)

    score = volume_ratio.rank("asset")

    score = score * is_liquid

    # --------------------------------------------------------
    # Select Strongest Assets
    # --------------------------------------------------------

    ranks = score.rank("asset")

    top_assets = xr.where(
        ranks >= 6,
        1,
        0
    )

    weights = score * top_assets

    # --------------------------------------------------------
    # Inverse Volatility Weighting
    # --------------------------------------------------------

    inv_vol = 1 / (vol20 + 1e-6)

    weights = weights * inv_vol

    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    denom = weights.sum("asset")

    weights = xr.where(
        denom > 0,
        weights / denom,
        0
    )

    return weights.fillna(0)

# ============================================================
# Generate Weights
# ============================================================

weights = strategy(data)

weights = qnout.clean(
    weights,
    data,
    "crypto_daily_long"
)

qnout.write(weights)

100% (15259192 of 15259192) |############| Elapsed Time: 0:00:00 Time:  0:00:000:00
Output cleaning...
Fix unique timestamps
Forward filling missing prices...
Check liquidity...
Ok.
Check for missed dates...
Ok.
Check positive positions...
Ok.
Final normalization...
Output cleaning complete.
Write output: /root/fractions.nc.gz

In [ ]:
# ============================================================
# Strategy Statistics
# ============================================================

import qnt.stats as qnstats

stats = qnstats.calc_stat(
    data,
    weights.sel(time=slice("2016-01-01", None))
)

stats_pd = stats.to_pandas()

print(stats_pd.tail())

print("\nFinal Metrics:")

print(
    stats_pd.iloc[-1][[
        "equity",
        "sharpe_ratio",
        "max_drawdown",
        "avg_turnover",
        "avg_holding_time"
    ]]
)